# 실습 5: 신분증 분석 (Identity Document Analysis)
**소요시간: 30분** | 난이도: ⭐⭐

## 학습 목표
1. `analyze_id` API로 여권, 운전면허증 등 신분증에서 정보를 추출합니다.
2. IdentityDocumentFields 구조를 파싱합니다.
3. 추출된 정보의 정규화 값(NormalizedValue)을 활용합니다.

## ⚠️ 개인정보 주의사항
- 실습에서는 **샘플/더미 신분증** 이미지를 사용하세요.
- 실제 개인 신분증 데이터는 실습 종료 후 즉시 삭제하세요.
- AWS에 업로드된 데이터는 Textract 처리 후 저장되지 않습니다.

## API 개요: analyze_id
```python
response = textract.analyze_id(
    DocumentPages=[        # 리스트로 전달 (최대 2페이지)
        {'Bytes': <바이트>}
    ]
)
```

### 응답 구조
```
response
└── IdentityDocuments[]
    └── IdentityDocumentFields[]
        ├── Type
        │   ├── Text          ← 필드명 (FIRST_NAME, LAST_NAME, DATE_OF_BIRTH 등)
        │   └── NormalizedValue.Value  ← 정규화된 타입
        └── ValueDetection
            ├── Text          ← 추출된 원본 텍스트
            ├── Confidence    ← 신뢰도
            └── NormalizedValue.Value  ← 정규화된 값 (날짜: YYYY-MM-DD 형식 등)
```

### 주요 필드 타입
| 타입 | 설명 | 정규화 예시 |
|---|---|---|
| `FIRST_NAME` | 이름 | - |
| `LAST_NAME` | 성 | - |
| `DATE_OF_BIRTH` | 생년월일 | YYYY-MM-DD |
| `EXPIRATION_DATE` | 만료일 | YYYY-MM-DD |
| `DOCUMENT_NUMBER` | 문서 번호 | - |
| `ID_TYPE` | 문서 유형 | PASSPORT, DRIVER_LICENSE |
| `STATE_IN_ADDRESS` | 주/지역 | - |
| `MRZ_CODE` | 기계 판독 영역 | - |


In [ ]:
# ✅ [제공 코드]
import boto3, os, json
from PIL import Image
import matplotlib.pyplot as plt

textract = boto3.client('textract', region_name='ap-northeast-2')

IMAGE_DIR = './images/'
image_path = os.path.join(IMAGE_DIR, 'lab05_id.jpg')

def load_document_bytes(path):
    with open(path, 'rb') as f:
        return f.read()

doc_bytes = load_document_bytes(image_path)

img = Image.open(image_path)
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.title('원본 신분증 (샘플)')
plt.show()


## ✏️ TODO 1: analyze_id API 호출

`analyze_id`는 `Document` 대신 **`DocumentPages`** (리스트)를 사용합니다.


In [ ]:
# ✏️ TODO 1: analyze_id API를 호출하세요
# 주의: Document가 아닌 DocumentPages (리스트)를 사용!
response = textract._____(                      # ← API 메서드명 (analyze_id)
    _____=[                                     # ← 'DocumentPages' (리스트!)
        {'Bytes': doc_bytes}
    ]
)

identity_docs = response[_____]                 # ← 'IdentityDocuments'
print(f"감지된 신분증 수: {len(identity_docs)}개")

id_doc = identity_docs[0]
fields = id_doc.get(_____,  [])                 # ← 'IdentityDocumentFields'
print(f"감지된 필드 수: {len(fields)}개")


## ✏️ TODO 2: 모든 필드 출력

추출된 모든 필드를 원본값과 정규화값을 함께 출력하세요.


In [ ]:
# ✏️ TODO 2: 모든 필드를 출력하세요
print("신분증 정보 추출 결과:")
print("=" * 60)

id_data = {}

for field in fields:
    field_name = field.get('Type', {}).get(_____, 'UNKNOWN')         # ← 'Text'
    raw_value  = field.get('ValueDetection', {}).get(_____, '')      # ← 'Text'
    confidence = field.get('ValueDetection', {}).get(_____, 0)       # ← 'Confidence'
    
    # 정규화된 값 (있는 경우)
    normalized = field.get('ValueDetection', {}).get('NormalizedValue', {})
    norm_value = normalized.get(_____, '')                            # ← 'Value'
    
    display_val = norm_value if norm_value else raw_value
    id_data[field_name] = display_val
    
    norm_tag = f" → 정규화: {norm_value}" if norm_value and norm_value != raw_value else ''
    print(f"  {field_name:<25}: {raw_value}{norm_tag} ({confidence:.1f}%)")


## ✏️ TODO 3: 만료일 유효성 검사

신분증 만료일을 파싱하여 현재 유효한지 확인하세요.


In [ ]:
# ✏️ TODO 3: 만료일을 파싱하여 현재 유효한지 검사하세요
from datetime import datetime, date

expiry_text = id_data.get(_____, '')       # ← 'EXPIRATION_DATE'

print(f"만료일 원본: {expiry_text}")

if expiry_text:
    try:
        # 정규화된 날짜 형식(YYYY-MM-DD) 파싱 시도
        expiry_date = datetime.strptime(_____, '%Y-%m-%d').date()  # ← expiry_text
        today = date.today()
        
        print(f"만료일: {expiry_date}")
        print(f"오늘 날짜: {today}")
        
        if expiry_date > _____:            # ← today
            days_left = (expiry_date - today).days
            print(f"\n✅ 유효한 신분증 (남은 기간: {days_left}일)")
        else:
            days_expired = (today - expiry_date).days
            print(f"\n❌ 만료된 신분증 (만료 후 {days_expired}일 경과)")
    except ValueError as e:
        print(f"날짜 파싱 오류: {e}")
        print("힌트: 정규화 값이 없는 경우 다른 날짜 형식을 시도해보세요")
else:
    print("만료일 정보를 찾을 수 없습니다.")


## ✏️ TODO 4: 추출 데이터 마스킹 처리

개인정보 보호를 위해 민감한 필드를 마스킹하여 출력하세요.


In [ ]:
# ✏️ TODO 4: 민감 필드를 마스킹하여 안전하게 출력하는 함수를 작성하세요
def mask_value(value, visible_chars=4):
    """
    'HONG GILDONG' → 'HONG ****DON'
    앞 visible_chars 글자만 보이고 나머지는 *로 마스킹
    """
    if not value or len(value) <= visible_chars:
        return value
    return value[:visible_chars] + '*' * (len(value) - visible_chars - 2) + value[-2:]

# 마스킹할 필드 목록
SENSITIVE_FIELDS = ['FIRST_NAME', 'LAST_NAME', 'DOCUMENT_NUMBER', 'MRZ_CODE']

print("[마스킹 처리된 신분증 정보]")
print("-" * 50)
for field_name, value in id_data.items():
    if field_name in _____:                   # ← SENSITIVE_FIELDS
        display = mask_value(_____,  4)       # ← value
    else:
        display = value
    print(f"  {field_name:<25}: {display}")


## 💡 심화 도전
1. 여권의 MRZ(Machine Readable Zone) 코드를 파싱하여 각 필드를 추출해보세요.
2. 앞면/뒷면 두 이미지를 `DocumentPages`에 함께 전달해보세요.
3. 추출된 정보를 DynamoDB에 저장하는 코드를 작성해보세요 (개인정보 암호화 포함).
